# Types

> Core internal types.

In [ ]:
#| default_exp types

## Imports

In [ ]:
#| export
from dataclasses import dataclass, field
from fastcore.utils import *
from fastspec.errors import *

In [ ]:
#| hide
from fastcore.test import *

The main philosophy of `fastllm` is to provide unified API to users where they can simply swap models/providers without code changes. The following types allow an internal canonical representation:

### Part

Canonical atomic content unit for multimodal inputs/outputs.

**Why it exists**
- Providers represent content blocks differently (`input_text`, `text`, `inlineData`, `source`, etc).
- `Part` gives one stable internal shape so serialization/parsing logic can stay at provider boundaries.

**Design Notes**
- `type` encodes semantic modality (`text`, `input_image`, `input_file`, etc).
- `text` is for simple text payloads.
- `data` carries structured provider-agnostic payload details when text is insufficient.

**Connections**
- Wrapped by `Msg.content`.
- Produced/consumed by `highlevel` coercion, provider serializers in `clients`, and normalizers in `normalize`.
- Key enabler for model-only swapping without rewriting message construction code.

All four providers represent content parts differently in their wire format:

- **[OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create)** — Content is an array of typed parts: `{"type": "input_text", "text": "..."}`, `{"type": "input_image", "image_url": "..."}`, `{"type": "input_audio", "input_audio": {"data": "...", "format": "..."}}`, `{"type": "input_file", "file_data": "data:application/pdf;base64,...", "filename": "..."}`
- **[OpenAI-compat (Chat)](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)** — Content is an array of typed parts: `{"type": "text", "text": "..."}`, `{"type": "image_url", "image_url": {"url": "..."}}`, `{"type": "input_audio", "input_audio": {"data": "...", "format": "..."}}`, `{"type": "file", "file": {"file_data": "data:application/pdf;base64,...", "filename": "..."}}`
- **[Anthropic](https://docs.anthropic.com/en/api/messages)** — Content blocks with `source` nesting: `{"type": "text", "text": "..."}`, `{"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": "..."}}`, `{"type": "document", "source": {"type": "base64", "media_type": "application/pdf", "data": "..."}}`
- **[Gemini](https://ai.google.dev/api/generate-content)** — A `parts` array of `{text}` or `{inlineData: {mimeType, data}}` or `{fileData: {mimeType, fileUri}}`: `{"text": "..."}`, `{"inlineData": {"mimeType": "image/jpeg", "data": "..."}}`, `{"fileData": {"mimeType": "application/pdf", "fileUri": "..."}}`

> **Spec sources:** OpenAI Responses `Input{Text,Image,File}Content` (`specs/openai.with-code-samples.yml:68093–68444`), OpenAI Chat `ChatCompletionRequestMessageContentPart*` (`specs/openai.with-code-samples.yml:35185–35321`), Anthropic `Request{Text,Image,Document}Block` (`specs/anthropic.yml:12159–12458`), Gemini `Part`/`Blob`/`FileData` (`specs/gemini.json:189–387`).

`Part` canonicalizes these into a uniform `(type, text, data)` triple. The canonical types, their aliases, and expected input formats:

| Canonical `Part.type` | Aliases accepted | Expected input format | `Part.text` | `Part.data` |
|---|---|---|---|---|
| `"text"` | — | Plain string | `"the text"` | `None` |
| `"input_image"` | `image`, `image_url` | URL, data URL (`data:image/...;base64,...`), or provider-native `source`/`inlineData` in `data` | URL shorthand | `{"url": "..."}` or `{"image_url": "..."}` or provider-native payload |
| `"input_audio"` | `audio` | base64 data + format (OpenAI), URL/`fileUri` (Gemini). Not supported on Anthropic. | URL shorthand | `{"input_audio": {"data": "...", "format": "wav"}}` or `{"inlineData": {...}}` |
| `"input_video"` | `video`, `video_url` | URL or `fileUri`. Currently Gemini only; OpenAI maps to `input_file`. Not supported on Anthropic. | URL shorthand | `{"video_url": "..."}` or `{"fileData": {...}}` |
| `"input_file"` | `file`, `pdf`, `document` | data URL, URL, `file_id`, or provider-native `source` in `data` | URL shorthand | `{"file_data": "data:...;base64,...", "filename": "..."}` or `{"source": {...}}` |

**Provider support matrix:**

| Canonical type | OpenAI Responses | OpenAI-compat (Chat) | Anthropic | Gemini |
|---|---|---|---|---|
| `text` | ✅ `input_text` | ✅ `text` | ✅ `text` | ✅ `text` |
| `input_image` | ✅ `input_image` | ✅ `image_url` | ✅ `image` + `source` | ✅ `inlineData` / `fileData` |
| `input_audio` | ✅ `input_audio` | ✅ `input_audio` | ❌ (raises `UnsupportedCapabilityError`) | ✅ `inlineData` / `fileData` |
| `input_video` | ✅ (mapped to `input_file`) | ✅ (mapped to `input_file`) | ❌ (raises `UnsupportedCapabilityError`) | ✅ `fileData` |
| `input_file` | ✅ `input_file` | ✅ `file` | ✅ `document` + `source` | ✅ `inlineData` / `fileData` |

**Escape hatch:** For any type, setting `Part.data["<provider>"]` (e.g. `Part.data["anthropic"]`) to a dict bypasses canonicalization and passes the payload directly to that provider (see `_provider_part`).

**Canonicalization design:** `Part.data` uses OpenAI-style key conventions as the single canonical input format. Users write one shape (e.g. `{"image_url": "..."}`, `{"file_data": "data:...;base64,..."}`, `{"input_audio": {"data": "...", "format": "wav"}}`), and the provider serializers (`_openai_responses_part`, `_anthropic_part`, `_gemini_part`) dispatch these into provider-native wire formats. This means users can swap models/providers without changing their `Part` construction code. `Part.text` serves as a URL shorthand fallback — e.g. `Part(type="input_image", text="https://example.com/img.png")` works when you just have a URL and no other metadata.

The key insight: `type` carries the **semantic modality**, not the provider-specific type name. Provider-specific payload details live in `data`, keeping `Part` itself provider-agnostic. This means downstream code can branch on `Part.type` without knowing which provider produced it.

In [ ]:
#| export
@dataclass(frozen=True)
class Part:
    "A normalized content part."
    type: str
    text: str = None
    data: dict = None

In [ ]:
#| export
PartType = str_enum('PartType', 'text', 'thinking', 'refusal', 'tool_use', 'server_tool_result', 'tool_result',
                    'input_image', 'input_audio', 'input_video', 'input_file')

### Msg

Canonical conversation turn abstraction.

**Why it exists**
- Conversation structures vary across APIs (chat messages, response input items, content blocks).
- `Msg` lets the rest of fastllm reason in one conversation format.

**Design Notes**
- `role` captures turn semantics (`user`, `assistant`, `tool`, etc).
- `content` is a list of `Part` to support multimodal turns consistently.
- `data` stores auxiliary metadata (tool call maps, provider-specific hints) without polluting canonical fields.

**Connections**
- Primary input to `acompletion` and provider clients.
- Used by toolloop replay flows (`StreamSummary`/`Completion` -> `Msg` coercion in `highlevel`).

All three providers structure conversation messages differently:

- **[OpenAI Chat](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)** — Messages are `{role, content}` objects. Roles: `system`, `developer`, `user`, `assistant`, `tool`. Content is a string or array of typed parts. Tool results use `role: "tool"` with `tool_call_id`.
- **[OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create)** — Input items with `{role, content}`. Text parts become `input_text`/`output_text` depending on role. Tool results are `{type: "function_call_output", call_id, output}`.
- **[Anthropic](https://docs.anthropic.com/en/api/messages)** — Messages are `{role, content}` with roles `user` or `assistant` only. System prompt is a separate top-level parameter. Tool results are `tool_result` content blocks inside `role: "user"` messages.
- **[Gemini](https://ai.google.dev/api/generate-content)** — Messages are `{role, parts}`. Roles: `user` or `model`. System prompt uses `system_instruction`. Tool results are `functionResponse` parts inside `role: "user"` messages.

> **Spec sources:** OpenAI `ChatCompletionRequest*Message` (`specs/openai.with-code-samples.yml:35022–35444`), Anthropic `InputMessage` + `RequestToolResultBlock` (`specs/anthropic.yml:11140–11159,12595–12652`), Gemini `Content` (`specs/gemini.json:171–188`).

`Msg` normalizes these into a canonical `(role, content, data)` triple:

| Canonical `Msg.role` | OpenAI Chat | OpenAI Responses | Anthropic | Gemini |
|---|---|---|---|---|
| `"system"` | `role: "system"` | `role: "system"` | Separate `system` param | `system_instruction` |
| `"user"` | `role: "user"` | `role: "user"` | `role: "user"` | `role: "user"` |
| `"assistant"` | `role: "assistant"` | `role: "assistant"` | `role: "assistant"` | `role: "model"` |
| `"tool"` | `role: "tool"` + `tool_call_id` | `function_call_output` + `call_id` | `role: "user"` + `tool_result` block | `role: "user"` + `functionResponse` |

**`Msg.data` metadata:** Provider-agnostic metadata that doesn't fit `role`/`content`:
- `tool_calls` — list of tool call dicts (assistant messages)
- `tool_call_id` / `call_id` / `id` — tool call identifier (tool result messages)
- `name` — tool function name (tool result messages)
- `is_error` — whether the tool result is an error (Anthropic-specific, forwarded)

**Escape hatch:** Like `Part`, setting `Msg.data["<provider>"]` (e.g. `Msg.data["anthropic"]`) to a dict with `"role"` bypasses serialization entirely and passes the raw message to that provider.

In [ ]:
#| export
@dataclass(frozen=True)
class Msg:
    "A normalized message."
    role: str
    content: List[Part]
    data: dict = None

### Caps (REMOVED)

Capability declaration for a client.

Let users handle

In [ ]:
# class Caps:
#     "Capability declaration for a client."
#     tools: bool = False
#     tool_choice: bool = False
#     streaming: bool = True
#     search: bool = False
#     reasoning: bool = False
#     prefill: bool = False
#     citations: bool = False
#     prompt_caching: bool = False
#     images: bool = True
#     pdfs: bool = False
#     url_context: bool = False

### RequestOptions

Unified high-level request option envelope.

**Why it exists**
- Each provider exposes overlapping but non-identical generation options.
- This type defines a canonical option surface with controlled escape hatches.

**Design Notes**
- Canonical knobs: `max_tokens`, `temperature`, `tools`, `tool_choice`, `reasoning_effort`
- Escape hatches: `native`, `extra_body`, `extra_headers`, `extra_query` for full provider coverage.

**Connections**
- Accepted by provider client methods and delegated through `acompletion`.
- Merged/translated in `clients` before request serialization.

Each provider maps generation options differently. `RequestOptions` defines canonical knobs that fastllm translates per-provider, plus escape hatches for full control.

**Canonical options and their provider mappings:**

| `RequestOptions` field | OpenAI Responses | OpenAI-compat (Chat) | Anthropic | Gemini |
|---|---|---|---|---|
| `max_tokens` | `max_output_tokens` | `max_tokens` | `max_tokens` (default 1024) | `generationConfig.maxOutputTokens` |
| `temperature` | `temperature` | `temperature` | `temperature` | `generationConfig.temperature` |
| `tools` | `_openai_responses_tools()` | `_openai_chat_tools()` | `_anthropic_tools()` | `_gemini_tools()` |
| `tool_choice` | passthrough | passthrough | `_anthropic_tool_choice()` | `_gemini_tool_choice()` |
| `reasoning_effort` | `{"reasoning": {"effort": ...}}` | `reasoning_effort` (passthrough) | `_anthropic_thinking()` | `_gemini_thinking_config()` |

**Escape hatches** — bypass canonicalization for provider-specific features:

| Field | Purpose | Example |
|---|---|---|
| `native` | Merged directly into the provider payload | `native={"store": True}` (OpenAI), `native={"context_management": {...}}` (Anthropic) |
| `extra_body` | Additional body params (merged after `native`) | `extra_body={"seed": 42}` |
| `extra_headers` | Additional HTTP headers | `extra_headers={"anthropic-beta": "..."}` |
| `extra_query` | Additional query parameters | `extra_query={"alt": "sse"}` |

**Design note:** Only options that are meaningfully canonical (supported by ≥2 providers with similar semantics) are first-class fields. Provider-specific features like `response_format` (OpenAI-only), `cache_control` (Anthropic content-block level via `Part.data`), and `cachedContent` (Gemini) go through escape hatches.

In [ ]:
#| export
@dataclass(frozen=True)
class RequestOptions:
    "Request options shared across providers."
    max_tokens: int = None
    temperature: float = None
    tools: List[dict] = None
    tool_choice  = None
    reasoning_effort: str = None
    native: dict = None
    extra_body: dict = None
    extra_headers: dict = None
    extra_query: dict = None

### Caching

Each provider handles prompt caching differently. fastllm does **not** provide a canonical `cache` option — instead, users pass provider-specific caching configuration through `Part.data` or `native`.

| Provider | Implicit (automatic) | Explicit (user-controlled) |
|---|---|---|
| **OpenAI** | ✅ Automatic for prompts ≥1024 tokens, no code changes needed | `native={"store": True}` for session storage; `prompt_cache_key`/`prompt_cache_retention` as routing hints |
| **Anthropic** | ❌ | `Part.data={"cache_control": {"type": "ephemeral"}}` on up to 4 content blocks per request |
| **Gemini** | ✅ Automatic on 2.5+ models (no cost saving guarantee) | `native={"cachedContent": "cachedContents/..."}` referencing a pre-created cache resource (outside fastllm scope) |

**Anthropic example** — place a cache breakpoint on a large system prompt:
```python
Msg(role="user", content=[
    Part(type="text", text=large_context, data={"cache_control": {"type": "ephemeral"}}),
    Part(type="text", text="Summarize the above"),
])
```
First request: `cache_creation_input_tokens` in usage. Subsequent requests: `cache_read_input_tokens` (75% discount on cached tokens).

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()